## Imports

In [1]:
import re
import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

from sentence_transformers import SentenceTransformer

from bertopic import BERTopic

import spacy
from spacy.tokens import Doc

from utilities import load_docs, save_docs

## Consts & Models

In [2]:
# Domain Normalization

NORMALIZE = {
    "közigazgatási": "közigazgatás",
    "bírósági": "bíróság",
    "alkotmányos": "alkotmány",
    "nemzeti": "nemzet",
    "hazafias": "nemzet",
    "szuverén": "szuverenitás",
    "független": "szuverenitás",
}


# CUSTOM STOPWORDS

CUSTOM_STOPWORDS = {
    "hogy",
    "amely",
    "ahol",
    "csak",
    "minden",
    "tehát",
    "ugye",
}

In [3]:
PROTOTYPES_PRO = [
    # sovereignty framing
    "A kormány megvédi Magyarország szuverenitását.",
    # emergency framing
    "Rendkívüli intézkedések szükségesek.",
    # anti-foreign interference
    "Külföldi érdekek beavatkoznak az ország ügyeibe.",
    # anti-judiciary framing
    "A bíróságok akadályozzák a kormány munkáját.",
    # centralized authority
    "Erős vezetésre van szükség.",
    # anti-pluralist framing
    "Csak a kormány képviseli a nemzet akaratát.",
]

In [4]:
PROTOTYPES_PRO_EA = [
    # sovereignty / external threat
    "A nemzeti szuverenitást minden eszközzel meg kell védeni.",
    "Külföldi szereplők próbálják befolyásolni az ország döntéseit.",
    "Brüsszel nem diktálhat Magyarországnak.",
    "A nemzet érdeke fontosabb a nemzetközi elvárásoknál.",
    # emergency exceptionalism
    "Válsághelyzetben gyors döntésekre van szükség.",
    "Rendkívüli idők rendkívüli intézkedéseket követelnek.",
    "A kormánynak szélesebb felhatalmazásra van szüksége a válság kezeléséhez.",
    "Krízis idején nem lehet a szokásos bürokratikus eljárásokhoz ragaszkodni.",
    # anti-judiciary / anti-checks-and-balances
    "A bíróságok akadályozzák a demokratikusan megválasztott kormány munkáját.",
    "Az alkotmányos intézmények nem béníthatják meg a végrehajtó hatalmat.",
    "A túlzott jogi kontroll lassítja az ország működését.",
    "A választott vezetőknek kell meghozniuk a végső döntéseket.",
    # centralization / concentration of power
    "Az ország hatékony működéséhez egységes irányítás szükséges.",
    "A stratégiai döntéseknek központi kézben kell maradniuk.",
    "Erős központi vezetés nélkül káosz alakul ki.",
    "A nemzeti érdek egységes politikai fellépést kíván.",
    # anti-pluralism
    "A kormány testesíti meg a nemzet akaratát.",
    "Az ellenzék akadályozza az ország fejlődését.",
    "A nemzeti egységet nem szabad pártpolitikai vitákkal gyengíteni.",
    "Aki a kormány ellen dolgozik, az az ország érdekei ellen dolgozik.",
    # anti-media / anti-civil society
    "A külföldről finanszírozott szervezetek veszélyeztetik a demokráciát.",
    "Az elfogult média akadályozza a nemzeti kormányzást.",
    "Az aktivista hálózatok politikai nyomást gyakorolnak az országra.",
    # soft technocratic centralization
    "A döntéshozatal hatékonysága fontosabb a hosszas intézményi vitáknál.",
    "A kormányzás nem bénulhat meg politikai viták miatt.",
    "A stabilitás érdekében bizonyos jogosítványokat központilag kell kezelni.",
]


PROTOTYPES_ANTI_EA = [
    # checks and balances
    "A hatalmi ágak szétválasztása alapvető demokratikus elv.",
    "A kormány hatalmát intézményi fékeknek kell korlátozniuk.",
    "A bíróságok függetlenségét minden körülmények között védeni kell.",
    "A demokratikus intézmények nem kerülhetnek politikai ellenőrzés alá.",
    # rule of law
    "A jogállamiság válsághelyzetben is megőrzendő.",
    "A rendkívüli intézkedéseknek is alkotmányos keretek között kell maradniuk.",
    "A törvények mindenkire egyformán vonatkoznak, a kormányra is.",
    # pluralism
    "A demokrácia alapja a politikai pluralizmus.",
    "Az ellenzék legitim szereplője a demokratikus rendszernek.",
    "Egyetlen párt sem sajátíthatja ki a nemzet képviseletét.",
    "A demokratikus vita erősíti az országot.",
    # media / civil society
    "A szabad sajtó nélkül nincs demokratikus elszámoltathatóság.",
    "A civil szervezetek fontos szerepet játszanak a demokráciában.",
    "A kritikus vélemények nem tekinthetők nemzetellenesnek.",
    # anti-centralization
    "A hatalom túlzott koncentrációja veszélyezteti a demokráciát.",
    "A döntéshozatal decentralizációja erősíti az intézményeket.",
    "Az állami intézményeknek függetlenül kell működniük a kormánytól.",
]


PROTOTYPES_NEUTRAL_GOVERNANCE = [
    "A parlament elfogadta az éves költségvetést.",
    "A kormány új közigazgatási reformot jelentett be.",
    "Az önkormányzatok további támogatásban részesülnek.",
    "A minisztérium nyilvánosságra hozta az új szabályozást.",
    "Az államigazgatás digitalizációja folytatódik.",
    "A képviselők vitát tartottak az új törvényjavaslatról.",
    "A választásokat jövő tavasszal rendezik meg.",
    "A kormány gazdaságélénkítő intézkedéseket vezet be.",
]

In [5]:
transformer_name = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"Loading transformer `{transformer_name}`...")
embedding_model = SentenceTransformer(transformer_name)

Loading transformer `paraphrase-multilingual-MiniLM-L12-v2`...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
spacy_model = "hu_core_news_md"

print(f"Loading HU SpaCy model `{spacy_model}`...")
nlp = spacy.load(spacy_model)

Loading HU SpaCy model `hu_core_news_md`...


## Preprocessing Functions

In [7]:
def normalize_token(token):
    if token.is_stop:
        return None

    if token.text.lower() in CUSTOM_STOPWORDS:
        return None

    if token.is_punct:
        return None

    if token.pos_ in {"DET", "CCONJ", "SCONJ", "AUX"}:
        return None

    # -------------------------
    # PROPER NOUNS
    # -------------------------

    if token.pos_ == "PROPN":
        form = token.text

    # -------------------------
    # VERBS
    # -------------------------

    elif token.pos_ == "VERB":
        form = token.text.lower()

    # -------------------------
    # ADJECTIVES
    # -------------------------

    elif token.pos_ == "ADJ":
        form = token.text.lower()

    # -------------------------
    # NOUNS
    # -------------------------

    elif token.pos_ == "NOUN":
        form = token.lemma_.lower()

    else:
        return None

    form = NORMALIZE.get(form, form)

    if len(form) < 2:
        return None

    return form

In [8]:
def normalize_doc(doc: Doc | str):
    tokens = []

    if isinstance(doc, str):
        doc = nlp(doc)

    for token in doc:
        norm = normalize_token(token)

        if norm:
            tokens.append(norm)

    return " ".join(tokens)

In [9]:
def preprocess_docs(docs: list[Doc]):
    normalized = []

    for doc in tqdm(docs):
        normalized.append(normalize_doc(doc))

    return normalized

In [10]:
def build_embeddings(texts):
    return embedding_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

In [11]:
from typing import Any


def retrieve_suspicious_sentences(
    original_sentences: list[Doc],
    normalized_sentences: list[Any],
    normalized_prototypes_list: list[list[str]],
    top_k=1000,
    min_score=0.7,
) -> list[list[dict[str, Any]]]:
    # create embeddings
    sentence_embeddings = build_embeddings(normalized_sentences)
    full_results: list[list[dict[str, Any]]] = []

    # for each list of normalized prototypes, calculate the scores
    for normalized_prototypes in normalized_prototypes_list:
        prototype_embeddings = build_embeddings(normalized_prototypes)

        # get similarity
        similarities = cosine_similarity(sentence_embeddings, prototype_embeddings)

        # find max similarity
        scores = similarities.max(axis=1)

        # build results
        results: list[dict[str, Any]] = []

        for i, score in enumerate(scores):
            if score >= min_score:
                results.append(
                    {
                        "sentence": original_sentences[i],
                        "normalized": normalized_sentences[i],
                        "ID": original_sentences[i]._.attrs["ID"],
                        "score": float(score),
                    }
                )

        results = sorted(results, key=lambda x: x["score"], reverse=True)
        full_results.append(results[:top_k])

    return full_results

In [12]:
def build_topics(normalized_sentences):
    topic_model = BERTopic(
        language="multilingual", calculate_probabilities=False, verbose=True
    )

    # TODO: Figure out whether I need `probs`` or not; replaced with _ for the time being
    topics, _ = topic_model.fit_transform(normalized_sentences)

    return topic_model, topics


def print_topics(topic_model, top_n=20):
    topic_info = topic_model.get_topic_info()

    print(topic_info.head(top_n))

In [13]:
def extract_keyphrases(texts, top_n=100):

    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)

    X = vectorizer.fit_transform(texts)

    features = vectorizer.get_feature_names_out()

    scores = np.asarray(X.mean(axis=0)).ravel()

    ranked = sorted(zip(features, scores), key=lambda x: x[1], reverse=True)

    return ranked[:top_n]

## Preprocessing

In [14]:
normalized_prototypes = [
    [normalize_doc(x) for x in prototypes]
    for prototypes in [
        PROTOTYPES_ANTI_EA,
        PROTOTYPES_NEUTRAL_GOVERNANCE,
        PROTOTYPES_PRO_EA,
    ]
]

In [18]:
print("Loading docs...")
docs = load_docs(nlp, ["2017"], suffix="md") or []

Loading docs...


0it [00:00, ?it/s]

In [22]:
# normalize sentences
normalized_sentences = preprocess_docs(docs)

  0%|          | 0/123267 [00:00<?, ?it/s]

100%|██████████| 123267/123267 [00:07<00:00, 15590.51it/s]


## Discovery

In [23]:
# retrieve suspicious
suspicious = retrieve_suspicious_sentences(
    docs, normalized_sentences, normalized_prototypes, top_k=-1
)

Batches:   0%|          | 0/3853 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
for results, type in zip(suspicious, ["CONTRA", "NEUTRAL", "PRO"]):
    # extract top suspicious sentences
    print(
        type,
        "\n",
        len(results),
        "\n",
        "\n".join([s["sentence"].text for s in results[:5]]),
    )

    # convert to pandas and save it
    df = pd.DataFrame(results)
    df["label"] = type
    df["sentence"] = df["sentence"].map(lambda x: x.text)
    df.to_csv(f"outputs/discovery_{type.lower()}-2017.csv")

CONTRA 
 920 
 A kormány jelen törvényjavaslata… a kormány?
Erről szól az alkotmányos berendezkedésünk.
Ha ez igaz, az már a bírói függetlenséget érintő kérdés.
Olyan szabályozások kerülnek át kormányrendeleti szintre, amelyek eddig törvényben voltak.
Szerintem ez mindenképpen befolyásolja bírói függetlenséget.
NEUTRAL 
 860 
 Csak nem azért, mert ezeknek az embereknek a szavazatára számítanak majd a jövő tavaszi választáson?
Bízom benne, hogy ez az összesen kettő szakaszból álló törvényjavaslat nem fog nagy vitát kiváltani.
A törvényjavaslat nagyon sok pontot érint.
Ezt a kérdést ugyancsak érinti a törvényjavaslat.
Ezekre a kérdésekre a törvényjavaslat mit sem mond.
PRO 
 2180 
 Mert nem hagytuk, hogy Brüsszelből egyre nagyobb nyomás alá helyezzék Magyarországot.
A kormány most ezen dolgozik.
Ma még nem látható, hogy ez segíti vagy nehezíti-e majd a magyar nemzeti érdekek brüsszeli érvényesítését.
Ezen dolgozik a kormány.
Tehát Magyarország Kormánya, hogyha a magyar parlament úgy dönt

In [ ]:
# get top results
print("\nTOP SUSPICIOUS SENTENCES:\n")

for x in suspicious[:20]:
    print(round(x["score"], 3), "|", x["sentence"])


TOP SUSPICIOUS SENTENCES:

0.954 | A magyar kormánynak az az elvi álláspontja, hogy minden magyar embernek joga van ahhoz, hogy a magyar állam megvédje.
0.938 | Mi a szuverén Magyarországot és lakóit védjük, ezért a magunk útját járjuk.
0.935 | A magyar kormány milyen lépéseket tesz, hogy megvédje az erdélyi magyarok alapjogait?
0.931 | Nekünk a középet kell tartani, a nemzet érdekét tartani, hiszen ezért van ma Magyarországnak egy nemzeti kormánya.
0.93 | Még egy ok arra, hogy megvédjük az eddigi eredményeket és kiálljunk Magyarország függetlensége mellett.
0.924 | Ez azt jelenti, hogy most egy újabb jogi vita fog kezdődni, amely során a magyar kormánynak harcolnia kell, hogy megvédje az országot.
0.919 | Számunkra a legfontosabb a magyar emberek biztonságának és Magyarország szuverenitásának megvédése.
0.919 | Ez Magyarország első számú követelése ma, a kormánynak pedig kötelessége meghallania a magyar emberek szavát.
0.909 | Magyarország eddig bizonyította, hogy képes megvédeni a p

In [ ]:
# do some topic modeling
topic_model, topics = build_topics(normalized_sentences)

# inspect the topics
print("\nDISCOVERED TOPICS:\n")
print_topics(topic_model)

2026-05-02 17:42:00,306 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3853 [00:00<?, ?it/s]

2026-05-02 17:44:05,331 - BERTopic - Embedding - Completed ✓
2026-05-02 17:44:05,332 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-02 17:51:12,859 - BERTopic - Dimensionality - Completed ✓
2026-05-02 17:51:12,877 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-02 17:51:35,133 - BERTopic - Cluster - Completed ✓
2026-05-02 17:51:35,189 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-02 17:51:39,152 - BERTopic - Representation - Completed ✓



DISCOVERED TOPICS:

    Topic  Count                                               Name  \
0      -1  48968                    -1_bíróság_magyar_javaslat_eset   
1       0   1542                            0_szó_elnök_köszönöm_úr   
2       1   1525        1_nyugdíj_nyugdíjas_szövetkezet_nyugdíjasok   
3       2   1104       2_országgyűlés_tisztelt_urr_elmegyógyintézet   
4       3   1068        3_kormányzat_kormány_kormányzati_kormányzás   
5       4    995                    4_ház_tisztelt_székház_tarolták   
6       5    995                     5_adós_hitel_bank_államadósság   
7       6    965              6_adó_adórendszer_adócsökkentés_adózó   
8       7    940                         7_szépen_elnök_köszönöm_úr   
9       8    915             8_ház_tisztelt_visszatérnének_mozzanat   
10      9    693    9_polgármester_önkormányzat_város_önkormányzati   
11     10    688  10_gazdaság_növekedés_versenyképesség_magyaror...   
12     11    661       11_bér_munkavállaló_munkanélkülis

In [ ]:
keyphrases = extract_keyphrases(normalized_sentences)

print("\nKEY PHRASES:\n")
for phrase, score in keyphrases[:50]:
    print(round(score, 5), phrase)


KEY PHRASES:

0.03117 tisztelt
0.02878 köszönöm
0.02505 úr
0.01766 szépen
0.01714 köszönöm szépen
0.01629 elnök
0.0157 elnök úr
0.01506 képviselőtárs
0.01385 szó
0.01252 tisztelt képviselőtárs
0.0116 magyar
0.01104 ház
0.01054 tisztelt ház
0.01009 év
0.00933 kormány
0.00927 ember
0.00899 államtitkár
0.00895 képviselő
0.00847 kérdés
0.00821 államtitkár úr
0.00821 forint
0.00769 köszönöm szó
0.00735 törvény
0.00705 javaslat
0.00691 országgyűlés
0.00676 szó elnök
0.00676 szó elnök úr
0.00667 figyelem
0.00639 tisztelt elnök
0.00635 tisztelt elnök úr
0.00633 ország
0.00628 eset
0.00617 tisztelt országgyűlés
0.00568 százalék
0.00565 lehetőség
0.00565 képviselő úr
0.00554 európai
0.00543 rendszer
0.00541 gondolom
0.00536 magyarországon
0.00524 helyzet
0.00521 köszönöm szó elnök
0.00514 tisztelt államtitkár
0.00509 fontos
0.00495 probléma
0.00491 érdek
0.00491 törvényjavaslat
0.00491 rész
0.00486 tisztelt államtitkár úr
0.00484 ügy
